# 24 — Pseudo-labeling v2 (pousser le nouveau champion)

nb23 a marché : pseudo-labeling → **LB 0.3573** (bat 08 à 0.3569). On pousse :
- **2 cycles** (self-training round 2)
- **seuils agressifs** (0.95/0.05 → plus de pseudo-labels sur les nouveaux comptes)

Pas de CV possible (labels test inconnus) → on génère les candidats, le **LB tranche**.
Garde 08 (0.3569) et 23 (0.3573) en fallback.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
from src import config as C
from src.utils import op03_mask, seed_everything, make_submission
from src.features.temporal import balance_features, recency_features
from src.features.behavioral import behavioral_features
from src.encoding import oof_target_encode_train, fit_target_map, apply_target_map, recent_target_rate
seed_everything(42)
DATA = ROOT / "data"
train = pd.read_csv(DATA / "train.csv"); test = pd.read_csv(DATA / "test.csv")
sample = pd.read_csv(DATA / "sample_submission.csv")
op03 = op03_mask(train).to_numpy(); y_all = train[C.TARGET].to_numpy()
te_op = op03_mask(test).to_numpy()
EPS = 1e-6; WINDOWS = (5, 10, 20); SM = 30

In [ ]:
def row_features(df):
    f = pd.DataFrame(index=df.index)
    f["amount_log1p"] = np.log1p(np.maximum(df[C.AMOUNT], 0))
    f["amount_vs_origin_before"] = df[C.AMOUNT] / (np.abs(df[C.ORIGIN_BAL_BEFORE]) + EPS)
    f["amount_vs_dest_before"] = df[C.AMOUNT] / (np.abs(df[C.DEST_BAL_BEFORE]) + EPS)
    f["origin_balance_before"] = df[C.ORIGIN_BAL_BEFORE]; f["dest_balance_before"] = df[C.DEST_BAL_BEFORE]
    return pd.concat([f, balance_features(df)], axis=1)
def base_build(df, ref):
    X = row_features(df).reset_index(drop=True)
    for col in [C.ORIGIN_ACCT, C.DEST_ACCT]:
        X[f"freq_{col}"] = df[col].map(ref[col].value_counts(normalize=True)).fillna(0).values
    beh = behavioral_features(df, ref).reset_index(drop=True)
    rec = recency_features(df, ref).reset_index(drop=True)
    rt = recent_target_rate(df, ref, C.ORIGIN_ACCT, C.PERIOD, C.TARGET, WINDOWS).reset_index(drop=True)
    return pd.concat([X, beh, rec, rt], axis=1)
def ftr(df, ref):
    X = base_build(df, ref); X["te_origin"] = oof_target_encode_train(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SM); return X
def fap(df, ref):
    X = base_build(df, ref); mp, gm = fit_target_map(ref, C.ORIGIN_ACCT, C.TARGET, smoothing=SM)
    X["te_origin"] = apply_target_map(df, C.ORIGIN_ACCT, mp, gm); return X
def make_cat():
    from catboost import CatBoostClassifier
    return CatBoostClassifier(loss_function="Logloss", eval_metric="PRAUC", depth=6,
                              learning_rate=0.05, iterations=600, random_seed=42, verbose=False)

test_op = test.iloc[np.where(te_op)[0]].copy()
def pseudo_augment(ref, proba, thr_f, thr_l):
    """Construit train + pseudo-labels du test (seuils thr_f/thr_l)."""
    mf = proba > thr_f; ml = proba < thr_l
    pse = test_op.iloc[np.where(mf | ml)[0]].copy()
    pse[C.TARGET] = (proba[mf | ml] > thr_f).astype(float)
    return pd.concat([ref, pse], ignore_index=True), int(mf.sum()), int(ml.sum())

## Candidat 1 : pseudo-labeling 2 cycles (seuils 0.98/0.02)

In [ ]:
ref0 = train.iloc[np.where(op03)[0]]; y0 = y_all[op03]
# cycle 0 : champion
m0 = make_cat().fit(ftr(ref0, ref0), y0)
p_test = m0.predict_proba(fap(test_op, ref0))[:, 1]
# cycle 1
aug1, nf1, nl1 = pseudo_augment(ref0, p_test, 0.98, 0.02)
m1 = make_cat().fit(ftr(aug1, aug1), aug1[C.TARGET].to_numpy())
p1 = m1.predict_proba(fap(test_op, aug1))[:, 1]
print(f"cycle1 pseudo: {nf1} fraudes / {nl1} legit")
# cycle 2 (re-pseudo avec m1)
aug2, nf2, nl2 = pseudo_augment(ref0, p1, 0.98, 0.02)
m2 = make_cat().fit(ftr(aug2, aug2), aug2[C.TARGET].to_numpy())
p2 = m2.predict_proba(fap(test_op, aug2))[:, 1]
print(f"cycle2 pseudo: {nf2} fraudes / {nl2} legit | corr c1-c2: {np.corrcoef(p1,p2)[0,1]:.4f}")
full = np.zeros(len(test)); full[te_op] = p2
path = make_submission(test[C.ID], full, "24_pseudo_2cycle")
print("écrit:", path)

## Candidat 2 : seuils agressifs (0.95/0.05), 1 cycle

In [ ]:
augA, nfA, nlA = pseudo_augment(ref0, p_test, 0.95, 0.05)
print(f"agressif pseudo: {nfA} fraudes / {nlA} legit (total {nfA+nlA})")
mA = make_cat().fit(ftr(augA, augA), augA[C.TARGET].to_numpy())
pA = mA.predict_proba(fap(test_op, augA))[:, 1]
full = np.zeros(len(test)); full[te_op] = pA
path = make_submission(test[C.ID], full, "24_pseudo_aggressive")
sub = pd.read_csv(path)
assert list(sub.columns) == ["id", "target"] and len(sub) == len(test)
assert set(sub["id"]) == set(sample["id"]) and sub["target"].between(0, 1).all()
print("écrit:", path, "| proba>0:", int((sub['target'] > 0).sum()))
print("\nCandidats à soumettre (le LB tranche) :")
print(" - 23_pseudo_blend50.csv (déjà généré, blend champion+pseudo)")
print(" - 24_pseudo_2cycle.csv")
print(" - 24_pseudo_aggressive.csv")